# 03 — Autoencoder training

**Production Pulse — Phase 4.** Complete training workflow for the dense autoencoder on the 52 TEP measurement features.

| Role | Source | Used for |
|---|---|---|
| train | fault-free training file | fitting the scaler and the autoencoder (**normal data only**) |
| validation | fault-free training file | early stopping only |
| calibration | fault-free testing file | choosing the alarm threshold only |
| evaluation | fault-free testing file | **final reporting only — never for tuning** |
| holdout | fault-free testing file | **reserved** for the live demo; not loaded or scored by this notebook. **Not fully unseen:** runs 42, 196, 241 and 304 were scored by the earlier, uncorrected notebook 02 run selection (see below) |

All splits are by whole simulation runs and seeded. No anomalous sample is ever used for training.

> **Disclosure (2026-10-21).** The first version of notebook 02 drew its runs from all 500 testing runs, ignoring the role split. With seed 42 it scored 4 holdout runs (**42, 196, 241, 304**) and 10 calibration runs. They were only scored, never used to fit, train or calibrate anything, but those 4 holdout runs are no longer unseen and are excluded from demo selection (`splitting.HOLDOUT_RUNS_PREVIOUSLY_SCORED`). Notebook 02 is now corrected. Details: `docs/RESEARCH_METHODOLOGY.md` §3.1.

⚠️ This notebook has **not been run yet**; it contains no outputs. Nothing here shows the model is effective until it has been executed on Windows with the `.venv313` kernel. Training writes a model to `artifacts/models/` (never next to the raw data).

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents)
                     if (p / "pyproject.toml").is_file() and (p / "src" / "production_pulse").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from production_pulse import get_paths, load_config, setup_logging
from production_pulse.workflow import prepare_fault_free_data
from production_pulse.autoencoder import AutoencoderDetector, build_autoencoder
from production_pulse.baselines import PCAReconstructionDetector, SingleFeatureThresholdDetector
from production_pulse.anomaly_generation import FAMILIES, AnomalyGenerator, AnomalySpec
from production_pulse.evaluation import HypothesisCriteria, assess_family_detectability, compare_detectors
from production_pulse.plots import plot_feature_errors, plot_score_distributions, plot_training_history, save_figure

paths = get_paths()
config = load_config(root=paths.root)
logger = setup_logging(config=config, paths=paths)

# ---- settings (model hyper-parameters live in configs/default.toml [autoencoder]) ----
SEED = config.random_seed
MODEL_DIR = "artifacts/models/autoencoder_v1"   # inside the safe output area
SAVE_ARTIFACTS = True                         # save model + metadata after training
OVERWRITE = False                             # True replaces an existing MODEL_DIR
SAVE_FIGURES = False                          # True -> reports/figures/
VERBOSE = 2                                   # Keras: 0 silent, 2 = one line per epoch
EVAL_RUNS_LIMIT = 60                          # evaluation runs used in section 5 (None = all)
ANOMALY_RATE = 0.05                           # same as notebook 02
CRITERIA = HypothesisCriteria()               # unchanged Phase 3 pass criteria
print("Autoencoder settings:", config.autoencoder.to_dict())

## 1. Prepare leakage-free data (scaler fitted on the train split only)

In [ ]:
data = prepare_fault_free_data(paths, config)
data.summary()

## 2. Model architecture

In [ ]:
build_autoencoder(len(data["train"].columns), config.autoencoder, seed=SEED).summary()

## 3. Train on normal training data; early stopping on the validation split

In [ ]:
detector = AutoencoderDetector(settings=config.autoencoder, seed=SEED, scaler=data.scaler, verbose=VERBOSE)
detector.fit(data["train"], data["validation"])
detector.training_summary_

In [ ]:
fig = plot_training_history(detector.history_, best_epoch=detector.training_summary_["best_epoch"])
if SAVE_FIGURES:
    save_figure(fig, "ae_training_history.png", paths, overwrite=True)
plt.show()

## 4. Calibrate the threshold on the separate calibration runs

In [ ]:
threshold = detector.calibrate(data["calibration"], target_fpr=config.evaluation.target_fpr)
print(detector.threshold_info())

fig = plot_score_distributions(
    {"validation (normal)": detector.score(data["validation"]), "calibration (normal)": detector.score(data["calibration"])},
    threshold=threshold, title="Reconstruction error on normal data")
if SAVE_FIGURES:
    save_figure(fig, "ae_normal_score_distribution.png", paths, overwrite=True)
plt.show()

## 5. Report on held-out evaluation runs — **report only, no tuning**
Do **not** change hyper-parameters or thresholds based on the numbers below (that would tune on evaluation data). Hyper-parameter studies belong to Phase 5 and must use train/validation/calibration data only.

The Phase 3 families are injected with the same generator settings and **unchanged pass criteria**. The rule check and PCA are refitted on the same train split and the PCA threshold is calibrated on the same calibration runs, so all three detectors see identical examples.

In [ ]:
eval_x, eval_meta = data["evaluation"], data.metadata["evaluation"]
if EVAL_RUNS_LIMIT is not None:
    runs = np.sort(np.random.default_rng(SEED).choice(eval_meta["simulationRun"].unique(), EVAL_RUNS_LIMIT, replace=False))
    keep = eval_meta["simulationRun"].isin(runs).to_numpy()
    eval_x, eval_meta = eval_x[keep], eval_meta[keep]

clean_ae = detector.detect(eval_x)
print(f"Autoencoder false-alarm rate on clean evaluation data: {clean_ae.predictions.mean():.4f} "
      f"(target on calibration: {config.evaluation.target_fpr})")

single = SingleFeatureThresholdDetector().fit(data["train"])
pca = PCAReconstructionDetector(n_components=0.90).fit(data["train"])
pca.calibrate(data["calibration"], target_fpr=config.evaluation.target_fpr)
gen = AnomalyGenerator().fit(data["train"])

In [ ]:
assessments, comparisons, injected = [], [], {}
for family in FAMILIES:
    r = gen.generate(eval_x, AnomalySpec(family, ANOMALY_RATE), seed=SEED, groups=eval_meta["simulationRun"])
    d_rule, d_pca, d_ae = single.detect(r.features), pca.detect(r.features), detector.detect(r.features)
    injected[family] = (r, d_ae)
    a = assess_family_detectability(r.row_info, d_rule, d_ae, features=r.features, events=r.events,
                                    clean_features=eval_x, criteria=CRITERIA)
    assessments.append(a)
    c = compare_detectors(r.labels, {"rule_check": d_rule, "pca": d_pca, "autoencoder": d_ae})
    c.insert(0, "family", family)
    comparisons.append(c)

assessment = pd.concat(assessments)
print("Multivariate detector in this table = AUTOENCODER. Verdicts are computed from these numbers.")
assessment[["in_range_fraction", "single_feature_recall", "single_feature_fpr", "multivariate_recall",
            "multivariate_fpr", "marginal_auc_excess", "marginally_hidden", "verdict", "reasons"]].round(4)

In [ ]:
comparison = pd.concat(comparisons).reset_index(names="detector").set_index(["family", "detector"])
comparison[["precision", "recall", "f1", "false_positive_rate", "pr_auc", "roc_auc"]].round(4)

In [ ]:
# Which sensors drive the reconstruction error inside one injected event?
PLOT_FAMILY = "correlation_break"
r, d_ae = injected[PLOT_FAMILY]
ev = r.events.iloc[0]
rows = r.features.iloc[ev["start_row"]: ev["end_row"] + 1]
print("event:", ev[["family", "group", "length", "anchor_feature", "modified_features"]].to_dict())
fig = plot_feature_errors(detector.reconstruction_errors(rows), top=12, title=f"Per-feature error, {PLOT_FAMILY} event 0")
plt.show()
lab = r.labels.to_numpy() == 1
fig = plot_score_distributions({"normal": d_ae.scores[~lab], PLOT_FAMILY: d_ae.scores[lab]},
                               threshold=detector.threshold_, title="Autoencoder score: normal vs injected")
if SAVE_FIGURES:
    save_figure(fig, f"ae_scores_{PLOT_FAMILY}.png", paths, overwrite=True)
plt.show()

## 6. Save the model with its scaler, threshold and metadata — then reload and verify

In [ ]:
if SAVE_ARTIFACTS:
    saved = detector.save(MODEL_DIR, paths, overwrite=OVERWRITE)
    for kind, path in saved.items():
        print(f"{kind:12s} -> {path.relative_to(paths.root)}")
    reloaded = AutoencoderDetector.load(MODEL_DIR, paths=paths)
    same = np.allclose(reloaded.score(data["calibration"]), detector.score(data["calibration"]), rtol=1e-6)
    print("Reloaded model reproduces calibration scores:", same, "| threshold:", reloaded.threshold_)
else:
    print("SAVE_ARTIFACTS is False - nothing written.")

## 6b. Save the measured evaluation as the report served by the API (`GET /metrics`)
Only values computed above are written. The file goes to `reports/evaluation_metrics.json` (safe output folder); the API reads it, never invents it.


In [ ]:
import json
from production_pulse.reporting import write_evaluation_report

SAVE_EVALUATION_REPORT = True
if SAVE_EVALUATION_REPORT:
    meta_file = paths.root / MODEL_DIR / "metadata.json"
    model_id = json.loads(meta_file.read_text(encoding="utf-8"))["model_sha256"][:12] if meta_file.is_file() else None
    report_path = write_evaluation_report(
        {
            "setup": {"seed": SEED, "anomaly_rate": ANOMALY_RATE, "eval_runs": int(eval_meta["simulationRun"].nunique()),
                      "eval_rows": int(len(eval_x)), "target_fpr": config.evaluation.target_fpr},
            "training": detector.training_summary_,
            "threshold": detector.threshold_info(),
            "clean_false_alarm_rate": float(clean_ae.predictions.mean()),
            "hypothesis_assessment_autoencoder": assessment.reset_index().to_dict(orient="records"),
            "detector_comparison": comparison.reset_index().to_dict(orient="records"),
        },
        paths, source="notebooks/03_autoencoder_training.ipynb", model_id=model_id)
    print("saved", report_path.relative_to(paths.root))

## 7. Record in the research logbook
- Training: epochs run, best epoch, best validation loss, whether early stopping triggered.
- Threshold and the false-alarm rate on clean evaluation data vs the calibration target.
- The assessment table: which families are SUPPORTED with the autoencoder, compared with PCA in notebook 02.
- Caveats: one seed, `EVAL_RUNS_LIMIT` runs; the holdout is **not fully unseen**: runs 42, 196, 241 and 304 were scored by the earlier (uncorrected) notebook 02 selection and must not be used in the demo. This notebook itself never loads the holdout runs. Keep the other holdout runs locked for the demo.